In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/03.Silver_Helper

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
sliver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
from pyspark.sql.functions import *

In [0]:
const_df = spark.read.table(bronze_table).filter(col('batch_id') == v_batch_id)



In [0]:
const_selected_df =(
    const_df.select("constructorId", "name", "nationality", "ingestion_timestamp", "source_files","batch_id")
    .withColumnRenamed("constructorId", "constructor_id")
    .withColumnRenamed("name", "constructor_name")
    .withColumn("nationality", initcap(col("nationality")))
)

In [0]:
const_distinct_df = const_selected_df.dropDuplicates(["constructor_id"])

In [0]:
columns_to_update=[
        "constructor_name",
        "nationality",
        "ingestion_timestamp",
        "source_files",
        "batch_id"
    ]

In [0]:
write_to_silver(input_df=const_distinct_df,target_table=sliver_table,
                merge_condn="t.constructor_id = s.constructor_id",
                columns_to_update=columns_to_update
                )

In [0]:
display(spark.read.table(sliver_table))